# PySR residual equation on top of the family-calibrated SPB baseline

**Input:** `features_with_folds.csv` from the m* calibration notebook (or `mstar_calibration.zip`).

**Changes from the XGBoost notebook:** mean electronegativity is **dropped** (it made held-out predictions worse), and
**Marcasite FeSb₂ is excluded** (SPB can't describe it: m* hits the search limit and the error is about 185 µV/K whatever the model).

**Prediction:** \|S\|_pred = S_SPB(n, T; family m*) + f(inputs). f is a single equation shared by all families, found by PySR.

**Everything stays leakage-free.** In each outer fold (the same paper-grouped folds as before):
1. family m* is refitted on the training papers → S_SPB for training and test papers
2. the training papers are split again: **fit papers (80 %)** and **validation papers (20 %)**. The equation
   complexity is chosen on the validation papers only, **never on the test papers**
3. every model is scored on the held-out test papers: **MAE, RMSE and R²**

| Model | Residual f |
|---|---|
| SPB (family m*) | 0 |
| SPB + linear correction | ridge regression on the inputs (the "simple correction" reference) |
| **SPB + PySR** | symbolic equation, complexity chosen on validation papers |
| SPB + XGBoost | gradient-boosted trees (flexible, not interpretable: an upper reference) |
| XGBoost direct | predicts \|S\| from the inputs without SPB (does physics help?) |

**Also in this notebook:** error vs equation complexity (Pareto curve), the chosen equation in each fold (is it stable?),
per-family results, parity and residual plots, and one final equation refitted on all data, reported with its
**cross-validated** accuracy.

## 0. Settings

In [ ]:
INPUT = "features_with_folds.csv"
OUT_DIR = "pysr_residual"
EXCLUDE_FAMILIES = ["Marcasite FeSb2"]
INPUTS = ["S_SPB", "T_K", "log10_n_cm3", "el_n_valence_mean", "dopant_total_frac"]   # electronegativity dropped
PYSR_SEEDS = None          # None = only the primary split (`fold`); or e.g. [42, 7, 123] to repeat (slow: ~5 folds × timeout each)
INNER_VAL_SHARE = 0.2      # share of TRAINING papers held back to choose the equation complexity
PARSIMONY_TOL = 0.01       # choose the SIMPLEST equation whose validation MAE is within 1 % of the best
MIN_TRAIN_PAPERS = 3
M_BOUNDS = (0.05, 20.0)

PYSR_PARAMS = dict(
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square", "sqrt", "log", "exp"],
    nested_constraints={"exp": {"exp": 0, "log": 0}, "log": {"exp": 0, "log": 0}, "sqrt": {"sqrt": 0}},
    maxsize=25, niterations=200, populations=24, population_size=40,
    elementwise_loss="L1DistLoss()",          # MAE-type loss, consistent with the m* fit and the reported metric
    batching=True, batch_size=512,
    timeout_in_seconds=240,                   # per fold; raise for a final run
    model_selection="best", progress=False, verbosity=0, random_state=0,
)
XGB_PARAMS = dict(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                  min_child_weight=5, reg_lambda=1.0, tree_method="hist", random_state=0, n_jobs=-1)

In [ ]:
# PySR needs Julia; the first import installs it (~3-5 min on Colab, once per runtime)
import sys, subprocess
try:
    import pysr  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysr"], check=True)
from pysr import PySRRegressor

In [ ]:
import json, zipfile, hashlib, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.special import expit
from scipy.optimize import minimize_scalar
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import xgboost as xgb
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 120)
OUT = Path(OUT_DIR); OUT.mkdir(exist_ok=True)
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
NEUTRAL, INK, INK2, GRID = "#b4b2ac", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})

## 1. Load, exclude, check inputs

In [ ]:
p = Path(INPUT)
if not p.exists():
    try:
        from google.colab import files
        p = Path(next(iter(files.upload())))
    except ImportError:
        raise FileNotFoundError(INPUT)
if p.suffix == ".zip":
    with zipfile.ZipFile(p) as z:
        z.extract("features_with_folds.csv", "."); p = Path("features_with_folds.csv")
d = pd.read_csv(p, low_memory=False)
n0 = len(d)
d = d[~d["family"].isin(EXCLUDE_FAMILIES)].reset_index(drop=True)
d["n_m3"] = 10 ** d["log10_n_cm3"] * 1e6
d["w"] = 1.0 / d.groupby("sample_key")["sample_key"].transform("size")
SEEDS = [int(c[6:]) for c in d.columns if c.startswith("fold_s")]
print(f"{len(d)} points ({n0 - len(d)} excluded: {EXCLUDE_FAMILIES}) | {d.sample_key.nunique()} samples | {d.SID.nunique()} papers")
print("inputs:", INPUTS, "| electronegativity dropped")

## 2. SPB baseline and m* calibration (same functions as before)

In [ ]:
KB, H_PL, M_E, E_CH = 1.380649e-23, 6.62607015e-34, 9.1093837015e-31, 1.602176634e-19
_eta = np.linspace(-30, 100, 2601)
_F = lambda j, e: quad(lambda x: x**j * expit(e - x), 0, max(e, 0) + 60, limit=200)[0]
_F12 = np.array([_F(0.5, e) for e in _eta]); _F1 = np.array([_F(1.0, e) for e in _eta]); _F0 = np.logaddexp(0, _eta)

def spb_seebeck_abs_uVK(n_m3, T_K, m_star=1.0):
    n_m3, T_K = np.asarray(n_m3, float), np.asarray(T_K, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        F12 = n_m3 / (4 * np.pi * (2 * m_star * M_E * KB * T_K / H_PL**2) ** 1.5)
        eta = np.interp(np.log(F12), np.log(_F12), _eta)
        lo, hi = F12 < _F12[0], F12 > _F12[-1]
        eta = np.where(lo, np.log(F12 * 2 / np.sqrt(np.pi)), eta)
        e_hi = (1.5 * F12) ** (2 / 3); e_hi = (1.5 * F12 / (1 + np.pi**2 / (8 * e_hi**2))) ** (2 / 3)
        eta = np.where(hi, e_hi, eta)
        S = np.where(lo, 2 - eta, np.where(hi, np.pi**2 / (3 * eta),
                     2 * np.interp(eta, _eta, _F1) / np.interp(eta, _eta, _F0) - eta))
    return S * KB / E_CH * 1e6

def fit_mstar(g):
    w = g["w"].to_numpy(); S, n, T = g["S_abs_uVK"].to_numpy(), g["n_m3"].to_numpy(), g["T_K"].to_numpy()
    loss = lambda lm: np.sum(w * np.abs(S - spb_seebeck_abs_uVK(n, T, np.exp(lm)))) / w.sum()
    return float(np.exp(minimize_scalar(loss, bounds=np.log(M_BOUNDS), method="bounded", options={"xatol": 1e-4}).x))

def calibrate(train, *others):
    m_glob = fit_mstar(train)
    fam = {f: fit_mstar(g) for f, g in train.groupby("family") if g["SID"].nunique() >= MIN_TRAIN_PAPERS}
    return [spb_seebeck_abs_uVK(df["n_m3"], df["T_K"], df["family"].map(fam).fillna(m_glob).to_numpy()) for df in (train,) + others]

def splits_from(fold_col):
    fold_col = np.asarray(fold_col)
    return [(np.where(fold_col != k)[0], np.where(fold_col == k)[0]) for k in sorted(np.unique(fold_col))]

def inner_split(train, share, seed=0):
    sids = sorted(train.SID.unique(), key=lambda s: hashlib.sha256(f"inner{seed}-{s}".encode()).hexdigest())
    val = set(sids[:max(1, int(round(share * len(sids))))])
    return ~train.SID.isin(val).to_numpy(), train.SID.isin(val).to_numpy()

def mae(y, p): return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))
def r2(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))

## 3. Input scaling for symbolic regression
PySR works best with inputs of order 1, so it sees rescaled variables with readable names.
The final equation is also converted back to physical units in section 7.

| PySR name | Definition |
|---|---|
| `s` | S_SPB / 100 µV/K |
| `t` | T / 300 K |
| `x` | log₁₀(n / cm⁻³) − 20 |
| `v` | mean valence electrons / 5 |
| `q` | dopant fraction × 100 (at.%) |

The target is the residual in µV/K.

In [ ]:
SR_NAMES = {"S_SPB": "s", "T_K": "t", "log10_n_cm3": "x", "el_n_valence_mean": "v", "dopant_total_frac": "q"}
SR_SCALE = {"S_SPB": (0, 100), "T_K": (0, 300), "log10_n_cm3": (20, 1), "el_n_valence_mean": (0, 5), "dopant_total_frac": (0, 0.01)}
def sr_matrix(df):
    return pd.DataFrame({SR_NAMES[c]: (df[c] - SR_SCALE[c][0]) / SR_SCALE[c][1] for c in INPUTS})

## 4. Cross-validated comparison
The time per fold is about `timeout_in_seconds`, plus a few seconds for the other models.

In [ ]:
fold_sets = [("primary", d["fold"])] + ([(f"s{sd}", d[f"fold_s{sd}"]) for sd in PYSR_SEEDS] if PYSR_SEEDS else [])
MODELS = ["SPB (family m*)", "SPB + linear", "SPB + PySR", "SPB + XGBoost", "XGBoost direct"]
rows, eq_rows, curve_rows = [], [], []
oof = {m: np.full(len(d), np.nan) for m in MODELS}           # out-of-fold predictions, primary split
t_all = time.time()
for split_name, fcol in fold_sets:
    for k, (tr, te) in enumerate(splits_from(fcol)):
        t0 = time.time()
        train, test = d.iloc[tr].copy(), d.iloc[te].copy()
        train["S_SPB"], test["S_SPB"] = calibrate(train, test)
        train["res"] = train.S_abs_uVK - train.S_SPB
        fit_m, val_m = inner_split(train, INNER_VAL_SHARE, seed=k)
        preds = {"SPB (family m*)": test.S_SPB.to_numpy()}
        lin = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(train[INPUTS], train.res, ridge__sample_weight=train.w)
        preds["SPB + linear"] = test.S_SPB + lin.predict(test[INPUTS])
        xr = xgb.XGBRegressor(**XGB_PARAMS).fit(train[INPUTS], train.res, sample_weight=train.w)
        preds["SPB + XGBoost"] = test.S_SPB + xr.predict(test[INPUTS])
        dir_in = [c for c in INPUTS if c != "S_SPB"]
        xd = xgb.XGBRegressor(**XGB_PARAMS).fit(train[dir_in], train.S_abs_uVK, sample_weight=train.w)
        preds["XGBoost direct"] = xd.predict(test[dir_in])
        # --- PySR: fit on fit-papers, choose complexity on validation papers ---
        sr = PySRRegressor(**PYSR_PARAMS)
        Xf, Xv, Xt = sr_matrix(train[fit_m]), sr_matrix(train[val_m]), sr_matrix(test)
        sr.fit(Xf, train.res[fit_m].to_numpy(), weights=train.w[fit_m].to_numpy())
        eqs = sr.equations_.reset_index(drop=True)
        val_mae = np.array([mae(train.res[val_m], sr.predict(Xv, index=i)) for i in range(len(eqs))])
        test_mae = np.array([mae(test.S_abs_uVK, test.S_SPB + sr.predict(Xt, index=i)) for i in range(len(eqs))])
        ok = np.isfinite(val_mae)
        best = np.nanmin(val_mae[ok])
        pick = int(np.where(ok & (val_mae <= best * (1 + PARSIMONY_TOL)))[0][0])     # simplest within tolerance
        preds["SPB + PySR"] = test.S_SPB + sr.predict(Xt, index=pick)
        for i in range(len(eqs)):
            curve_rows.append({"split": split_name, "fold": k, "complexity": int(eqs.loc[i, "complexity"]),
                               "val_MAE_residual": val_mae[i], "test_MAE": test_mae[i]})
        eq_rows.append({"split": split_name, "fold": k, "complexity": int(eqs.loc[pick, "complexity"]),
                        "equation": str(eqs.loc[pick, "equation"]), "val_MAE_residual": val_mae[pick],
                        "test_MAE": test_mae[pick]})
        for m in MODELS:
            y, pr = test.S_abs_uVK.to_numpy(), np.asarray(preds[m])
            rows.append({"split": split_name, "fold": k, "model": m, "MAE": mae(y, pr), "RMSE": rmse(y, pr), "R2": r2(y, pr),
                         "R2_residual": r2(y - test.S_SPB, pr - test.S_SPB)})
            if split_name == "primary":
                oof[m][te] = pr
        print(f"[{split_name}] fold {k}: PySR complexity {eq_rows[-1]['complexity']}, "
              f"test MAE SPB {mae(test.S_abs_uVK, test.S_SPB):.1f} -> PySR {mae(test.S_abs_uVK, preds['SPB + PySR']):.1f} "
              f"({time.time() - t0:.0f} s)")
res = pd.DataFrame(rows); eqs_chosen = pd.DataFrame(eq_rows); curve = pd.DataFrame(curve_rows)
print(f"total {time.time() - t_all:.0f} s")

## 5. Results
**Two views:** the mean ± sd over folds (fold-to-fold variation), and "pooled" metrics computed once on all
out-of-fold predictions of the primary split. R² is 1 − SS_res/SS_tot on \|S\|. `R2_residual` is the share of
the *SPB residual's* variance that the correction explains (≈ 0 for SPB alone, slightly negative when the residual has a
non-zero mean; negative for a correction = worse than no correction). Per-family R² on few points is unreliable.

In [ ]:
summ = res.groupby("model")[["MAE", "RMSE", "R2", "R2_residual"]].agg(["mean", "std"]).reindex(MODELS)
print("Per-fold mean ± sd (µV/K for MAE and RMSE):"); display(summ.round(3))
y_all = d.S_abs_uVK.to_numpy(); spb_oof = oof["SPB (family m*)"]
pooled = pd.DataFrame({m: {"MAE": mae(y_all, oof[m]), "RMSE": rmse(y_all, oof[m]), "R2": r2(y_all, oof[m]),
                           "R2_residual": r2(y_all - spb_oof, oof[m] - spb_oof)} for m in MODELS}).T
print("Pooled out-of-fold, primary split:"); display(pooled.round(3))
# paired comparison per fold: does PySR beat the baselines on every fold?
piv = res.pivot_table(index=["split", "fold"], columns="model", values="MAE")
for ref in ["SPB (family m*)", "SPB + linear", "SPB + XGBoost"]:
    diff = piv["SPB + PySR"] - piv[ref]
    print(f"PySR − {ref}: mean ΔMAE {diff.mean():+.2f} µV/K, PySR better in {int((diff < 0).sum())}/{len(diff)} folds")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 2.9), sharey=True)
y = np.arange(len(MODELS))[::-1]
cols = [NEUTRAL, CAT[3], CAT[0], CAT[1], CAT[6]]
for a, met, lab in zip(ax, ["MAE", "RMSE", "R2"], ["MAE (µV/K)", "RMSE (µV/K)", "R² on |S|"]):
    mu, sd = summ[(met, "mean")].to_numpy(), summ[(met, "std")].to_numpy()
    a.barh(y, mu, xerr=sd, color=cols, height=0.55, error_kw=dict(ecolor=INK2, lw=1, capsize=3))
    for yi, v in zip(y, mu):
        a.text(v, yi + 0.33, f"{v:.3f}" if met == "R2" else f"{v:.1f}", fontsize=8, color=INK, ha="center")
    a.set_xlabel(lab); a.xaxis.grid(True, color=GRID, lw=0.6); a.set_axisbelow(True)
    if met == "R2": a.set_xlim(min(0, mu.min() - 0.05), 1)
ax[0].set_yticks(y, MODELS)
fig.suptitle("Held-out papers: mean ± sd over folds", x=0.01, ha="left", fontsize=11, color=INK)
plt.tight_layout(); plt.savefig(OUT / "fig_model_comparison.png"); plt.show()

### Error vs equation complexity (Pareto)
The validation curve is what chose the complexity. The test curve is shown only for **description**: picking the
minimum of the test curve would leak the test data into model selection.

In [ ]:
cv_ = curve.groupby("complexity")[["val_MAE_residual", "test_MAE"]].agg(["mean", "std", "count"])
cv_ = cv_[cv_[("test_MAE", "count")] >= max(2, curve.fold.nunique() // 2)]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].errorbar(cv_.index, cv_[("val_MAE_residual", "mean")], yerr=cv_[("val_MAE_residual", "std")], color=CAT[0],
               marker="o", ms=4, lw=1.5, capsize=2)
ax[0].set(xlabel="equation complexity", ylabel="validation MAE of residual (µV/K)", title="Validation papers (used to choose)")
ax[1].errorbar(cv_.index, cv_[("test_MAE", "mean")], yerr=cv_[("test_MAE", "std")], color=CAT[1], marker="o", ms=4, lw=1.5, capsize=2)
spb_line = summ.loc["SPB (family m*)", ("MAE", "mean")]; xgb_line = summ.loc["SPB + XGBoost", ("MAE", "mean")]
ax[1].axhline(spb_line, color=NEUTRAL, ls="--", lw=1); ax[1].text(cv_.index.max(), spb_line, " SPB", va="center", fontsize=8, color=INK2)
ax[1].axhline(xgb_line, color=CAT[1], ls=":", lw=1); ax[1].text(cv_.index.max(), xgb_line, " SPB+XGB", va="center", fontsize=8, color=INK2)
ax[1].set(xlabel="equation complexity", ylabel="test MAE of |S| (µV/K)", title="Test papers (description only)")
for a in ax: a.yaxis.grid(True, color=GRID, lw=0.6); a.set_axisbelow(True)
for c in eqs_chosen.complexity: ax[0].axvline(c, color=INK2, lw=0.5, alpha=0.4)
plt.tight_layout(); plt.savefig(OUT / "fig_complexity_curve.png"); plt.show()

### Chosen equation in each fold: is the structure stable?
The same variables and a similar form in every fold means a robust result. Very different equations mean the data
only weakly constrain the correction.

In [ ]:
display(eqs_chosen)
used = {v: eqs_chosen.equation.str.contains(rf"\b{v}\b").mean() for v in SR_NAMES.values()}
print("Share of folds whose chosen equation uses each variable:",
      {k: f"{used[SR_NAMES[k]]:.0%}" for k in INPUTS})

### Per-family results (primary split, pooled out-of-fold)

In [ ]:
fam_rows = []
for g, gg in d.groupby("group"):
    idx = gg.index.to_numpy(); yy = gg.S_abs_uVK.to_numpy()
    fam_rows.append({"group": g, "papers": gg.SID.nunique(), "points": len(gg),
                     **{f"MAE {m}": mae(yy, oof[m][idx]) for m in MODELS},
                     "R2 SPB+PySR": r2(yy, oof["SPB + PySR"][idx]) if len(gg) > 5 else np.nan})
fam_tab = pd.DataFrame(fam_rows).set_index("group").sort_values("papers", ascending=False)
fam_tab["PySR better than SPB"] = fam_tab["MAE SPB + PySR"] < fam_tab["MAE SPB (family m*)"]
display(fam_tab.round(2))
print("by carrier type:")
display(pd.DataFrame([{"carrier_type": c, "points": len(gg), **{f"MAE {m}": mae(gg.S_abs_uVK, oof[m][gg.index]) for m in MODELS}}
                      for c, gg in d.groupby("carrier_type")]).set_index("carrier_type").round(1))

### Parity and residual plots (primary split, out-of-fold)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 4))
lim = [0, 800]
for a, m, c in zip(ax[:2], ["SPB (family m*)", "SPB + PySR"], [NEUTRAL, CAT[0]]):
    a.scatter(oof[m], y_all, s=3, alpha=0.3, color=c, rasterized=True)
    a.plot(lim, lim, color=INK2, ls="--", lw=0.8); a.set(xlim=lim, ylim=lim, xlabel="predicted |S| (µV/K)", ylabel="measured |S| (µV/K)")
    a.set_title(f"{m}\nMAE {mae(y_all, oof[m]):.1f}  RMSE {rmse(y_all, oof[m]):.1f}  R² {r2(y_all, oof[m]):.3f}", loc="left", fontsize=9)
for a, col, lab in zip(ax[2:], ["T_K", "log10_n_cm3"], ["T (K)", "log₁₀ n (cm⁻³)"]):
    a.scatter(d[col], y_all - spb_oof, s=3, alpha=0.2, color=NEUTRAL, label="SPB residual", rasterized=True)
    a.scatter(d[col], y_all - oof["SPB + PySR"], s=3, alpha=0.2, color=CAT[0], label="after PySR", rasterized=True)
    a.axhline(0, color=INK2, lw=0.8); a.set(xlabel=lab, ylabel="measured − predicted (µV/K)", ylim=(-400, 400))
    a.legend(markerscale=4, frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "fig_parity_residuals.png"); plt.show()

## 6. Final equation (refitted on all data)
m* is calibrated on all papers and PySR is refitted on all points. From its hall of fame we take the equation whose
complexity is the **median of the complexities chosen in the CV folds**. That rule is fixed in advance, not tuned on test data.
**Its expected accuracy on new papers is the cross-validated result from section 5**, not its fit on these data.

In [ ]:
d["S_SPB"] = calibrate(d)[0]; d["res"] = d.S_abs_uVK - d.S_SPB
final = PySRRegressor(**{**PYSR_PARAMS, "timeout_in_seconds": PYSR_PARAMS["timeout_in_seconds"] * 2})
final.fit(sr_matrix(d), d.res.to_numpy(), weights=d.w.to_numpy())
hof = final.equations_.reset_index(drop=True)
target_c = int(np.median(eqs_chosen.complexity))
i_final = int((hof.complexity - target_c).abs().idxmin())
print(f"target complexity {target_c} -> chosen complexity {int(hof.loc[i_final, 'complexity'])}")
display(hof[["complexity", "loss", "equation"]])
eq_final = str(hof.loc[i_final, "equation"])
print("\nFinal residual equation (scaled variables):\n  ΔS [µV/K] =", eq_final)
try:
    import sympy as sp
    expr = final.sympy(i_final)
    Ssp, T, lgn, val, dop = sp.symbols("S_SPB T log10n Nval dop")
    phys = sp.simplify(expr.subs({sp.Symbol("s"): Ssp / 100, sp.Symbol("t"): T / 300, sp.Symbol("x"): lgn - 20,
                                  sp.Symbol("v"): val / 5, sp.Symbol("q"): dop / 0.01}))
    print("In physical units (S_SPB in µV/K, T in K, log10n = log10(n/cm^-3), Nval = mean valence e-, dop = dopant fraction):")
    print("  ΔS [µV/K] =", phys)
    print("LaTeX:", sp.latex(phys))
except Exception as e:
    phys = None; print("sympy conversion skipped:", e)
d["S_pred_final_fit"] = d.S_SPB + final.predict(sr_matrix(d), index=i_final)
print(f"In-sample fit of the final equation (optimistic, NOT the reported accuracy): "
      f"MAE {mae(d.S_abs_uVK, d.S_pred_final_fit):.1f}, R² {r2(d.S_abs_uVK, d.S_pred_final_fit):.3f}")

## 7. Save

In [ ]:
res.to_csv(OUT / "cv_metrics_per_fold.csv", index=False); summ.to_csv(OUT / "cv_metrics_summary.csv")
pooled.to_csv(OUT / "cv_metrics_pooled_primary.csv"); eqs_chosen.to_csv(OUT / "chosen_equations_per_fold.csv", index=False)
curve.to_csv(OUT / "complexity_curve.csv", index=False); fam_tab.to_csv(OUT / "per_family_metrics.csv")
hof.drop(columns=[c for c in hof.columns if c in ("lambda_format", "sympy_format")], errors="ignore").to_csv(OUT / "final_hall_of_fame.csv", index=False)
out_pred = d[["point_id", "SID", "sample_key", "family", "group", "T_K", "log10_n_cm3", "S_abs_uVK"]].copy()
for m in MODELS: out_pred[f"oof_{m}"] = oof[m]
out_pred.to_csv(OUT / "oof_predictions_primary.csv", index=False)
report = {"settings": {"INPUTS": INPUTS, "EXCLUDE_FAMILIES": EXCLUDE_FAMILIES, "PYSR_PARAMS": {k: str(v) for k, v in PYSR_PARAMS.items()},
                       "INNER_VAL_SHARE": INNER_VAL_SHARE, "PARSIMONY_TOL": PARSIMONY_TOL, "splits": [s for s, _ in fold_sets]},
          "cv_mean": summ.xs("mean", axis=1, level=1).round(3).to_dict(), "cv_sd": summ.xs("std", axis=1, level=1).round(3).to_dict(),
          "pooled_primary": pooled.round(3).to_dict(),
          "final_equation_scaled": eq_final, "final_equation_physical": str(phys) if phys is not None else None,
          "variable_definitions": {"s": "S_SPB/100 uV/K", "t": "T/300 K", "x": "log10(n/cm^-3)-20", "v": "mean valence/5", "q": "dopant fraction*100"}}
(OUT / "pysr_report.json").write_text(json.dumps(report, indent=2, default=float))
import shutil
shutil.make_archive("pysr_residual", "zip", OUT)
try:
    from google.colab import files
    files.download("pysr_residual.zip")
except ImportError:
    print("zip at pysr_residual.zip")